# **Project Name**    - **Shopper Spectrum: Customer Segmentation and Product Recommendations in E-Commerce**



##### **Project Type**    - Unsupervised Machine Learning (Customer Segmentation & Recommendation System) with EDA
##### **Contribution**    - Individual
##### **Name** - Priyanshi Solanki

# **Project Summary -**

The rapid growth of e-commerce platforms has resulted in the generation of massive volumes of transactional data, creating opportunities for businesses to better understand customer purchasing behavior and improve customer engagement. This project, **Shopper Spectrum: Customer Segmentation and Product Recommendations in E-Commerce**, focuses on analyzing retail transaction data to derive meaningful business insights through customer segmentation and personalized product recommendations.

The project begins with comprehensive data preprocessing, including handling missing values, removing cancelled transactions, and filtering invalid purchase records. Exploratory Data Analysis (EDA) is then performed to identify purchasing trends, top-selling products, customer spending patterns, and country-wise transaction distributions.

To understand **customer behavior, Recency, Frequency, and Monetary (RFM)** metrics are calculated for each customer. These metrics are standardized and used to segment customers through clustering techniques such as K-Means. The optimal number of customer segments is determined using the Elbow Method and Silhouette Score. Based on purchasing behavior, customers are categorized into groups such as High-Value, Regular, Occasional, and At-Risk customers.

In addition to customer segmentation, the project implements an item-based collaborative filtering recommendation system. Product similarities are computed using cosine similarity on customer-product purchase matrices, enabling personalized product recommendations. Given a product, the system recommends the top five similar products frequently purchased by customers.

Finally, an interactive Streamlit application is developed, allowing users to generate product recommendations and predict customer segments in real time. The solution helps businesses improve targeted marketing strategies, customer retention, inventory planning, and overall customer experience.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


The e-commerce industry generates enormous amounts of customer transaction data every day. However, many businesses struggle to transform this raw data into actionable insights that can improve customer engagement, retention, and sales performance.

The objective of this project is to analyze online retail transaction data and identify meaningful customer segments using **Recency, Frequency, and Monetary (RFM) analysis**. By applying clustering techniques, customers can be grouped based on their purchasing behavior, enabling businesses to design targeted marketing campaigns and retention strategies.

Additionally, the project aims to develop a personalized product recommendation system using **item-based collaborative filtering**. By analyzing historical purchasing patterns and product similarities, the system recommends relevant products to customers, enhancing their shopping experience and increasing the likelihood of future purchases.

The final solution integrates customer segmentation and product recommendation capabilities into an interactive Streamlit application that provides real-time business insights and decision support for e-commerce organizations.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Date & Time
from datetime import datetime

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

# Model Saving
import pickle

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

### Dataset Loading

In [ ]:
# Load Dataset

df = pd.read_csv("online_retail.csv", encoding='latin1')

print("Dataset Loaded Successfully")

### Dataset First View

In [ ]:
# Dataset First Look

df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count

print(f"Number of Rows : {df.shape[0]}")
print(f"Number of Columns : {df.shape[1]}")

### Dataset Information

In [ ]:
# Dataset Info

df.info()

#### Duplicate Values

In [ ]:
# Duplicate Values

duplicate_count = df.duplicated().sum()

print("Total Duplicate Rows :", duplicate_count)

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values

missing_df = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage': ((df.isnull().sum() / len(df)) * 100).round(2)
})

missing_df = missing_df.sort_values(by='Missing Values', ascending=False)

missing_df

In [ ]:
plt.figure(figsize=(8,5))

missing_df['Missing Values'].plot(kind='bar')

plt.title('Missing Values Count')
plt.ylabel('Count')
plt.xlabel('Columns')
plt.xticks(rotation=45)

plt.show()

### What did you know about your dataset?

• The dataset contains retail transaction records from an online e-commerce business.

• It consists of 8 variables namely InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, and Country.

• Each row represents a purchased product within a transaction. The dataset contains customer-level and product-level information that can be utilized for customer segmentation and recommendation system development.

• Initial exploration revealed missing values primarily in CustomerID and Description, which must be handled during preprocessing.

• InvoiceDate provides temporal information required for Recency calculation in RFM analysis, while Quantity and UnitPrice enable Monetary value computation. CustomerID serves as the unique identifier for customer segmentation.

• The dataset is well-suited for unsupervised learning techniques such as clustering and collaborative filtering-based recommendation systems.

In [ ]:
df.columns.tolist()

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns

for col in df.columns:
    print(col)

In [ ]:
# Dataset Describe

df.describe(include='all')

### Variables Description

| Variable | Description |
|-----------|-------------|
| InvoiceNo | Unique invoice/transaction number generated for each purchase |
| StockCode | Unique code assigned to each product |
| Description | Name or description of the product |
| Quantity | Number of units purchased in a transaction |
| InvoiceDate | Date and time when the transaction occurred |
| UnitPrice | Price of a single product unit |
| CustomerID | Unique identifier assigned to each customer |
| Country | Country from which the customer made the purchase |

### Understanding Variable Types

| Variable | Data Type | Category |
|-----------|-----------|-----------|
| InvoiceNo | Object | Categorical |
| StockCode | Object | Categorical |
| Description | Object | Categorical |
| Quantity | Integer | Numerical |
| InvoiceDate | Object (will be converted to Datetime) | Date-Time |
| UnitPrice | Float | Numerical |
| CustomerID | Float | Identifier |
| Country | Object | Categorical |

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable

for col in df.columns:
    print(f"\n{col}")
    print("-"*50)
    print("Unique Values :", df[col].nunique())

In [ ]:
# Sample Unique Values

for col in df.columns:
    print(f"\n{col}")
    print(df[col].dropna().unique()[:10])

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Create a copy of dataset

df_clean = df.copy()

# Initial Shape
print("Original Shape:", df_clean.shape)

# Remove rows with missing CustomerID
df_clean = df_clean.dropna(subset=['CustomerID'])

# Remove cancelled invoices
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Remove negative or zero Quantity
df_clean = df_clean[df_clean['Quantity'] > 0]

# Remove negative or zero UnitPrice
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Create Total Amount feature
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Reset index
df_clean.reset_index(drop=True, inplace=True)

print("Cleaned Shape:", df_clean.shape)

# Display first few rows
df_clean.head()

In [ ]:
# Missing Values after cleaning

df_clean.isnull().sum()

# Check for negative values

print("Negative Quantity:", (df_clean['Quantity'] <= 0).sum())
print("Negative UnitPrice:", (df_clean['UnitPrice'] <= 0).sum())

# Check cancelled invoices

df_clean[df_clean['InvoiceNo'].astype(str).str.startswith('C')].shape

### What all manipulations have you done and insights you found?

### Data Wrangling Summary

•  A copy of the original dataset was created to preserve the raw data.

•  Rows containing missing CustomerID values were removed because customer identification is essential for customer segmentation and recommendation analysis.

•  Cancelled transactions were excluded by removing invoices whose InvoiceNo begins with the letter 'C'.

•  Records with zero or negative Quantity values were removed since they do not represent valid purchases.

•  Records with zero or negative UnitPrice values were removed to ensure accurate monetary calculations.

•  InvoiceDate was converted into datetime format to enable time-based analysis and RFM feature engineering.

•  A new feature named TotalAmount was created by multiplying Quantity and UnitPrice. This feature represents the total revenue generated from each transaction.

### Key Insights

•  A significant number of records contained missing CustomerID values and were removed.

•  Cancelled transactions were present in the dataset and needed to be excluded to prevent distortion of customer purchasing behavior.

•  Invalid purchase records with negative quantities and prices were identified and removed.

•  The cleaned dataset is now suitable for Exploratory Data Analysis, RFM customer segmentation, clustering, and product recommendation system development.

•  The newly created TotalAmount feature will be used in Monetary value calculation during RFM analysis.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1 (Top 10 Countries by Revenue)

In [ ]:
# Chart - 1 visualization code
# Top 10 Countries by Revenue

country_revenue = df_clean.groupby('Country')['TotalAmount'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12,6))
sns.barplot(x=country_revenue.values, y=country_revenue.index)
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Total Revenue')
plt.ylabel('Country')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart was selected because it clearly compares total revenue across different countries. Since country names can be long, a horizontal layout improves readability and makes it easier to identify the highest revenue-generating markets.

##### 2. What is/are the insight(s) found from the chart?

The chart shows that the United Kingdom contributes the highest revenue compared to other countries. This indicates that most customer transactions and sales revenue are concentrated in the UK market. Other countries contribute comparatively smaller revenue, showing that international sales are present but less dominant.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Impact**-
Yes, this insight can create positive business impact because the business can focus more on its strongest market while also identifying countries with growth potential. Since the United Kingdom generates the highest revenue, marketing campaigns, loyalty programs, and inventory planning can be prioritized for this region.

**Negative Impact**- Over dependence on a single country may lead to negative growth if demand decreases in that market. Therefore, the business should also work on improving sales in other countries to reduce market concentration risk.

#### Chart - 2 (Top 10 Countries by Transactions)

In [ ]:
# Chart - 2 visualization code
# Top 10 Countries by Number of Transactions

country_transactions = df_clean.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False).head(10)

plt.figure(figsize=(12,6))
sns.barplot(x=country_transactions.values, y=country_transactions.index)
plt.title('Top 10 Countries by Number of Transactions')
plt.xlabel('Number of Transactions')
plt.ylabel('Country')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart was chosen to compare the number of transactions across countries. This chart is suitable because it ranks countries based on transaction volume and helps identify markets with high customer activity.

##### 2. What is/are the insight(s) found from the chart?

The chart indicates that the United Kingdom has the highest number of transactions. This confirms that the UK is not only the highest revenue-generating country but also the most active customer market. Other countries have significantly fewer transactions, suggesting lower customer engagement or smaller market reach.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Impact**- Yes, this insight helps the business understand where customer engagement is strongest. Countries with a high number of transactions can be targeted with customer retention strategies, personalized promotions, and loyalty programs to further increase customer lifetime value.

**Negative Impact**- The concentration of transactions in a few countries indicates that customer activity is unevenly distributed. If customer engagement decreases in these key markets, the business may experience a significant decline in transaction volume and overall sales performance.

#### Chart - 3 (Top 10 Best-Selling Products)

In [ ]:
# Chart - 3 visualization code
# Top 10 Best-Selling Products by Quantity

top_products_quantity = df_clean.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12,6))
sns.barplot(x=top_products_quantity.values, y=top_products_quantity.index)
plt.title('Top 10 Best-Selling Products by Quantity')
plt.xlabel('Total Quantity Sold')
plt.ylabel('Product Description')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart was selected because it effectively displays product-wise sales quantity. Product names are usually long, so a horizontal chart makes the labels easier to read and helps identify the most frequently purchased products.

##### 2. What is/are the insight(s) found from the chart?

The chart highlights the products with the highest total quantity sold. These products are the most popular among customers and represent strong demand in the online retail store. Such products can be considered key inventory items because they are purchased frequently.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Impact**- Yes, identifying best-selling products enables the business to maintain sufficient inventory levels, prevent stock shortages, and design promotional campaigns around high-demand products. These products can also be bundled with complementary items to increase sales revenue.

**Negative Impact**- Excessive focus on only best-selling products may cause the business to neglect slow-moving products. This can result in inventory imbalance, increased holding costs, and reduced visibility for products that may have future growth potential.

#### Chart - 4 (Top 10 Revenue-Generating Products)

In [ ]:
# Chart - 4 visualization code
# Top 10 Revenue-Generating Products

top_products_revenue = df_clean.groupby('Description')['TotalAmount'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12,6))
sns.barplot(x=top_products_revenue.values, y=top_products_revenue.index)
plt.title('Top 10 Revenue-Generating Products')
plt.xlabel('Total Revenue')
plt.ylabel('Product Description')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart was selected to compare products based on total revenue contribution. This chart helps identify products that generate the highest monetary value for the business.

##### 2. What is/are the insight(s) found from the chart?

The chart shows the products contributing the highest revenue. Some products may generate high revenue because of large sales volume, while others may contribute more due to higher unit prices. These products are financially important for the business and can influence revenue growth.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Impact**- Yes, this insight helps the business identify products that contribute the most revenue. These products can be prioritized in marketing campaigns, recommendation systems, and inventory planning to maximize profitability and revenue growth.

**Negative Impact**- Heavy reliance on a limited number of high-revenue products may increase business risk. If demand for these products declines due to competition, seasonality, or changing customer preferences, overall revenue may be significantly affected.

#### Chart - 5 (Monthly Sales Trend)

In [ ]:
# Chart - 5 visualization code
# Monthly Sales Trend

monthly_sales = df_clean.resample('M', on='InvoiceDate')['TotalAmount'].sum()

plt.figure(figsize=(12,6))
monthly_sales.plot(marker='o')

plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.grid(True)

plt.show()

##### 1. Why did you pick the specific chart?

A line chart was selected because it is the most effective visualization for showing trends over time. It helps identify increases, decreases, and seasonal patterns in monthly sales revenue.

##### 2. What is/are the insight(s) found from the chart?

The chart reveals how revenue changes across different months. It helps identify peak sales periods, seasonal demand fluctuations, and months with relatively lower performance. Such patterns are valuable for understanding customer purchasing behavior over time.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Impact**- Yes, identifying seasonal sales trends enables the business to plan inventory, marketing campaigns, and staffing requirements more effectively. Peak demand periods can be leveraged through promotional activities to maximize revenue.

**Negative Impact**- Months with declining sales may indicate reduced customer engagement or seasonal slowdowns. If these periods are ignored, the business may face reduced revenue and inventory inefficiencies during low-demand seasons.

#### Chart - 6 (Daily Transaction Trend)

In [ ]:
# Chart - 6 visualization code
# Daily Transactions Trend

daily_transactions = df_clean.groupby(df_clean['InvoiceDate'].dt.date)['InvoiceNo'].nunique()

plt.figure(figsize=(14,6))
daily_transactions.plot()

plt.title('Daily Transaction Trend')
plt.xlabel('Date')
plt.ylabel('Number of Transactions')
plt.grid(True)

plt.show()

##### 1. Why did you pick the specific chart?

A line chart was chosen because it effectively displays transaction activity over time and helps identify spikes, drops, and fluctuations in customer purchasing behavior.

##### 2. What is/are the insight(s) found from the chart?

The chart shows variations in daily transaction volume. Certain days experience significantly higher customer activity, while others record fewer transactions. This provides insight into purchasing frequency and operational demand.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Impact-** Understanding daily transaction patterns allows the business to optimize staffing, logistics, and inventory management. High-traffic days can be targeted with promotions and marketing campaigns to maximize customer engagement.

**Negative Impact**- Significant fluctuations in transaction volume may indicate inconsistent customer activity. Sudden declines in daily transactions could negatively affect revenue generation and operational planning if not addressed promptly.

#### Chart - 7 (Quantity Distribution)

In [ ]:
# Chart - 7 visualization code
# Quantity Distribution

plt.figure(figsize=(10,6))

sns.histplot(df_clean['Quantity'], bins=50, kde=True)

plt.title('Distribution of Product Quantity Purchased')
plt.xlabel('Quantity')
plt.ylabel('Frequency')

plt.show()

##### 1. Why did you pick the specific chart?

A histogram was selected because it effectively displays the distribution of purchased quantities and helps identify common purchasing patterns and potential outliers.

##### 2. What is/are the insight(s) found from the chart?

The distribution typically shows that most transactions involve small purchase quantities, while a smaller number of transactions contain very large quantities. This indicates that the majority of customers make modest purchases, with a few bulk-buying customers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Impact**- Understanding purchase quantity patterns helps improve inventory management and demand forecasting. The business can maintain optimal stock levels and identify bulk-purchasing customers for special offers and loyalty programs.

**Negative Impact**- Extremely large quantity purchases may create inventory pressure if stock replenishment is not managed effectively. Overestimating demand based on a few large transactions can also lead to excess inventory costs.

#### Chart - 8 (Revenue Distribution (TotalAmount))

In [ ]:
# Chart - 8 visualization code
# Revenue Distribution

plt.figure(figsize=(10,6))

sns.histplot(df_clean['TotalAmount'], bins=50, kde=True)

plt.title('Revenue Distribution per Transaction')
plt.xlabel('Transaction Revenue')
plt.ylabel('Frequency')

plt.show()

##### 1. Why did you pick the specific chart?

A histogram was chosen because it helps visualize how transaction revenue is distributed and whether customer spending is concentrated in lower-value or higher-value purchases.

##### 2. What is/are the insight(s) found from the chart?

The chart generally shows that most transactions generate relatively small revenue amounts, while a limited number of transactions contribute significantly higher revenue. This indicates a right-skewed spending distribution.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Impact**- Revenue distribution analysis helps identify spending behavior and customer value segments. High-value transactions can be studied further to develop premium customer retention and upselling strategies.

**Negative Impact**- A strong dependence on a small number of high-value transactions may expose the business to revenue volatility. Losing a few high-spending customers could significantly impact overall revenue performance.

#### Chart - 9 (Top 10 Customers by Spending)

In [ ]:
# Chart - 9 visualization code
# Top 10 Customers by Spending

top_customers = df_clean.groupby('CustomerID')['TotalAmount'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12,6))

sns.barplot(
    x=top_customers.index.astype(str),
    y=top_customers.values
)

plt.title('Top 10 Customers by Total Spending')
plt.xlabel('Customer ID')
plt.ylabel('Total Spending')

plt.xticks(rotation=45)

plt.show()

##### 1. Why did you pick the specific chart?

A bar chart was selected because it effectively compares spending behavior across multiple customers and clearly identifies the highest-value customers contributing to business revenue.

##### 2. What is/are the insight(s) found from the chart?

The chart identifies customers who contribute the highest revenue to the business. A small number of customers often account for a significant portion of total sales, indicating the presence of high-value customers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Impact**- Yes, identifying top-spending customers enables the business to implement loyalty programs, personalized recommendations, exclusive discounts, and premium services. Retaining these customers can significantly improve long-term revenue and customer lifetime value.

**Negative Impact**- Heavy dependence on a small group of high-spending customers may create business risk. If these customers stop purchasing or move to competitors, overall revenue could decline substantially.

#### Chart - 10 (RFM Distribution)

Feature Creation Before Visualization

In [ ]:
# Create RFM Dataset

snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

RFM Distribution Plot

In [ ]:
# Chart - 10 visualization code
# RFM Distribution

fig, axes = plt.subplots(1,3, figsize=(18,5))

sns.histplot(rfm['Recency'], bins=30, kde=True, ax=axes[0])
axes[0].set_title('Recency Distribution')

sns.histplot(rfm['Frequency'], bins=30, kde=True, ax=axes[1])
axes[1].set_title('Frequency Distribution')

sns.histplot(rfm['Monetary'], bins=30, kde=True, ax=axes[2])
axes[2].set_title('Monetary Distribution')

plt.tight_layout()

plt.show()

##### 1. Why did you pick the specific chart?

Histograms were selected because they effectively display the distribution of Recency, Frequency, and Monetary values, helping understand customer purchasing behavior before clustering.

##### 2. What is/are the insight(s) found from the chart?

The distributions reveal variations in customer behavior. Some customers purchase frequently and spend large amounts, while others purchase rarely and contribute less revenue. These differences justify the need for customer segmentation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Impact**- Understanding RFM distributions helps identify valuable customers, regular customers, occasional buyers, and at-risk customers. Businesses can design personalized marketing campaigns for each segment to improve retention and revenue.

**Negative Impact**- If customer segments with low frequency and low monetary value continue to grow, overall business profitability may decline. Ignoring at-risk customers can also increase customer churn over time.

#### Chart - 11 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code

numeric_df = df_clean[['Quantity','UnitPrice','TotalAmount']]

plt.figure(figsize=(8,6))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap')

plt.show()

##### 1. Why did you pick the specific chart?

A heatmap was selected because it clearly visualizes the strength and direction of relationships among numerical variables. It helps identify which variables are strongly associated with revenue generation.

##### 2. What is/are the insight(s) found from the chart?

The heatmap shows the correlation between Quantity, UnitPrice, and TotalAmount. TotalAmount generally exhibits a positive relationship with Quantity and UnitPrice because revenue is calculated using both variables. Strong correlations indicate variables that significantly influence transaction value.

#### Chart - 12 - Pair Plot

In [ ]:
# Pair Plot visualization code

sample_df = df_clean[['Quantity','UnitPrice','TotalAmount']].sample(
    min(5000, len(df_clean)),
    random_state=42
)

sns.pairplot(sample_df)

plt.show()

##### 1. Why did you pick the specific chart?

A pair plot was selected because it simultaneously visualizes relationships among multiple numerical variables. It helps identify patterns, trends, clusters, and potential outliers within the dataset.

##### 2. What is/are the insight(s) found from the chart?

The pair plot reveals how Quantity, UnitPrice, and TotalAmount interact with one another. It helps identify positive relationships, data concentration regions, and unusual observations that may affect future modeling and clustering performance.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Hypothesis testing is performed to validate whether the observed patterns in customer purchasing behavior are statistically significant or occurred by chance. Based on the exploratory data analysis, three hypotheses were formulated related to customer spending, purchasing frequency, and country-wise revenue contribution. Appropriate statistical tests were applied to determine whether significant differences exist among customer groups and business segments.

### Hypothetical Statement - 1

"Customers with high purchase frequency spend significantly more money than customers with low purchase frequency."

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H₀)-
There is no significant difference in monetary spending between high-frequency customers and low-frequency customers.

Alternate Hypothesis (H₁)-
There is a significant difference in monetary spending between high-frequency customers and low-frequency customers.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

from scipy.stats import ttest_ind

# High and Low Frequency Groups

median_frequency = rfm['Frequency'].median()

high_freq = rfm[rfm['Frequency'] > median_frequency]['Monetary']
low_freq = rfm[rfm['Frequency'] <= median_frequency]['Monetary']

# Independent T-Test

t_stat, p_value = ttest_ind(high_freq, low_freq)

print("T-Statistic :", t_stat)
print("P-Value :", p_value)

if p_value < 0.05:
    print("Reject Null Hypothesis")
else:
    print("Fail to Reject Null Hypothesis")

##### Which statistical test have you done to obtain P-Value?

Independent Two Sample T-Test

##### Why did you choose the specific statistical test?

The Independent T-Test is appropriate because the objective is to compare the average monetary spending between two independent customer groups (high-frequency customers and low-frequency customers). The test determines whether the difference in spending behavior is statistically significant.

### Hypothetical Statement - 2

"Average transaction revenue differs significantly among the top revenue-generating countries."

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H₀)-
There is no significant difference in average transaction revenue among the selected countries.

Alternate Hypothesis (H₁)-
There is a significant difference in average transaction revenue among the selected countries.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

from scipy.stats import f_oneway

top_countries = df_clean['Country'].value_counts().head(5).index

country_data = []

for country in top_countries:
    country_data.append(
        df_clean[df_clean['Country'] == country]['TotalAmount']
    )

f_stat, p_value = f_oneway(*country_data)

print("F-Statistic :", f_stat)
print("P-Value :", p_value)

if p_value < 0.05:
    print("Reject Null Hypothesis")
else:
    print("Fail to Reject Null Hypothesis")

##### Which statistical test have you done to obtain P-Value?

One-Way ANOVA Test

##### Why did you choose the specific statistical test?

One-Way ANOVA is used because the comparison involves more than two independent groups (countries). The test evaluates whether there is a statistically significant difference in average transaction revenue across multiple countries.

### Hypothetical Statement - 3

"Customers who purchased recently have significantly higher purchase frequency than customers who have not purchased recently."

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H₀)-
There is no significant difference in purchase frequency between recent customers and less recent customers.

Alternate Hypothesis (H₁)-
There is a significant difference in purchase frequency between recent customers and less recent customers.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

from scipy.stats import ttest_ind

median_recency = rfm['Recency'].median()

recent_customers = rfm[rfm['Recency'] <= median_recency]['Frequency']
old_customers = rfm[rfm['Recency'] > median_recency]['Frequency']

t_stat, p_value = ttest_ind(recent_customers, old_customers)

print("T-Statistic :", t_stat)
print("P-Value :", p_value)

if p_value < 0.05:
    print("Reject Null Hypothesis")
else:
    print("Fail to Reject Null Hypothesis")

##### Which statistical test have you done to obtain P-Value?

Independent Two Sample T-Test

##### Why did you choose the specific statistical test?

The Independent T-Test is appropriate because it compares purchase frequency between two independent customer groups categorized based on recency. The test determines whether recent customers purchase significantly more frequently than less active customers.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation

print("Missing Values Before Cleaning:")
print(df.isnull().sum())

print("\nMissing Values After Cleaning:")
print(df_clean.isnull().sum())

#### What all missing value imputation techniques have you used and why did you use those techniques?

Missing values were primarily found in the CustomerID column.

Instead of applying statistical imputation techniques such as mean, median, or mode imputation, rows containing missing CustomerID values were removed using row deletion.

This approach was selected because CustomerID is a critical feature for customer segmentation and collaborative filtering. Records without customer identification cannot be associated with any customer and therefore cannot contribute to RFM analysis or recommendation generation.

As a result, removing these records ensured data quality and prevented inaccurate customer behavior analysis.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier Treatments

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
sns.boxplot(x=df_clean['Quantity'])
plt.title("Quantity")

plt.subplot(1,3,2)
sns.boxplot(x=df_clean['UnitPrice'])
plt.title("Unit Price")

plt.subplot(1,3,3)
sns.boxplot(x=df_clean['TotalAmount'])
plt.title("Total Amount")

plt.tight_layout()
plt.show()

##### What all outlier treatment techniques have you used and why did you use those techniques?

Outliers were identified using boxplots and distribution analysis.

Although outliers were present in Quantity, UnitPrice, and TotalAmount variables, they were not removed because they represent genuine customer purchasing behavior rather than data entry errors.

Large purchases and high-spending customers are important business observations and play a significant role in identifying premium customer segments during RFM analysis.

Therefore, outliers were retained to preserve valuable business information and maintain the integrity of customer segmentation results.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns

categorical_columns = ['InvoiceNo','StockCode','Description','Country']

for col in categorical_columns:
    print(f"{col} : {df_clean[col].nunique()} unique values")

#### What all categorical encoding techniques have you used & why did you use those techniques?

Traditional categorical encoding techniques such as Label Encoding and One-Hot Encoding were not required for this project.

Customer segmentation was performed using numerical RFM features, while the recommendation system was built using pivot tables and collaborative filtering.

Categorical variables such as CustomerID, StockCode, Description, and Country were used directly during aggregation and recommendation generation without requiring encoding.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features

# Total Transaction Value

df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Snapshot Date

snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# RFM Features

rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting

selected_features = rfm[['Recency','Frequency','Monetary']]

selected_features.head()

##### What all feature selection methods have you used  and why?

Domain knowledge and business relevance were used as the primary feature selection method.

RFM variables were selected because they are widely accepted indicators of customer purchasing behavior and are highly effective for customer segmentation tasks.

Only Recency, Frequency, and Monetary features were retained to reduce noise and improve clustering performance.

##### Which all features you found important and why?

**Recency:**
Measures how recently a customer made a purchase. Recent customers are more likely to engage with future promotions.

**Frequency:**
Measures how often a customer purchases products. Frequent customers indicate stronger engagement and loyalty.

**Monetary:**
Measures total customer spending. Customers with high monetary value contribute significantly to business revenue.

These three features collectively provide a complete representation of customer behavior and are essential for effective customer segmentation.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

Yes.

The RFM variables operate on different scales. Monetary values can be significantly larger than Frequency and Recency values.

To ensure that no feature dominates the clustering process, data transformation through standardization was performed before applying K-Means clustering.

In [ ]:
# Transform Your Data

rfm.describe()

### 6. Data Scaling

In [ ]:
# Scaling your data

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(
    rfm[['Recency','Frequency','Monetary']]
)

rfm_scaled[:5]

##### Which method have you used to scale you data and why?

StandardScaler was used for feature scaling.

K-Means clustering relies on Euclidean distance calculations and is highly sensitive to differences in feature magnitudes.

StandardScaler transforms features to have zero mean and unit variance, ensuring that Recency, Frequency, and Monetary contribute equally during cluster formation.

## ***7. ML Model Implementation***

### ML Model - 1 (**K-Means Clustering**)

In [ ]:
# ML Model - 1 Implementation
# K-Means Clustering

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

# Fit the Algorithm
kmeans.fit(rfm_scaled)

# Predict on the model
rfm['KMeans_Cluster'] = kmeans.predict(rfm_scaled)

rfm.head()

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

K-Means Clustering is an unsupervised machine learning algorithm used to group customers based on similar purchasing behavior. In this project, K-Means was applied on RFM features: Recency, Frequency, and Monetary.

The model divides customers into 4 clusters based on distance from cluster centroids. Customers within the same cluster have similar buying patterns.

The model performance was evaluated using Inertia and Silhouette Score. Lower inertia indicates compact clusters, while a higher silhouette score indicates better separation between clusters.

In [ ]:
# Visualizing evaluation Metric Score chart

inertia_scores = []
silhouette_scores = []

k_range = range(2, 11)

for k in k_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(rfm_scaled)

    inertia_scores.append(model.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, labels))

evaluation_df = pd.DataFrame({
    'K': list(k_range),
    'Inertia': inertia_scores,
    'Silhouette Score': silhouette_scores
})

evaluation_df

In [ ]:
# Elbow Method Chart

plt.figure(figsize=(10,5))
plt.plot(evaluation_df['K'], evaluation_df['Inertia'], marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
# Silhouette Score Chart

plt.figure(figsize=(10,5))
plt.plot(evaluation_df['K'], evaluation_df['Silhouette Score'], marker='o')
plt.title('Silhouette Score for Different K Values')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques

best_k = evaluation_df.sort_values(by='Silhouette Score', ascending=False).iloc[0]['K']
best_k = int(best_k)

print("Best number of clusters based on Silhouette Score:", best_k)

best_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)

# Fit the Algorithm
best_kmeans.fit(rfm_scaled)

# Predict on the model
rfm['Final_Cluster'] = best_kmeans.predict(rfm_scaled)

print("Final KMeans Inertia:", best_kmeans.inertia_)
print("Final Silhouette Score:", silhouette_score(rfm_scaled, rfm['Final_Cluster']))

##### Which hyperparameter optimization technique have you used and why?

Elbow Method and Silhouette Score analysis were used for hyperparameter optimization.

The main hyperparameter tuned was the number of clusters. Since K-Means is an unsupervised learning algorithm, traditional GridSearchCV is not suitable because there is no target variable. Therefore, different values of K were tested and evaluated using inertia and silhouette score.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, improvement was observed by selecting the number of clusters that produced better cluster separation and compactness. The optimized model provides more meaningful customer segments compared to selecting the number of clusters randomly.

#####Cluster Profiling

In [ ]:
# Cluster Profiling

cluster_profile = rfm.groupby('Final_Cluster')[['Recency','Frequency','Monetary']].mean()
cluster_profile

In [ ]:
# Visualize Cluster Profiles

cluster_profile.plot(kind='bar', figsize=(12,6))

plt.title('Customer Cluster Profiles Based on RFM Values')
plt.xlabel('Cluster')
plt.ylabel('Average Value')
plt.xticks(rotation=0)
plt.grid(True)

plt.show()

In [ ]:
# Cluster Labeling Function

def assign_segment(row):
    if row['Recency'] <= rfm['Recency'].quantile(0.50) and row['Frequency'] >= rfm['Frequency'].quantile(0.75) and row['Monetary'] >= rfm['Monetary'].quantile(0.75):
        return 'High-Value'
    elif row['Frequency'] >= rfm['Frequency'].quantile(0.50) and row['Monetary'] >= rfm['Monetary'].quantile(0.50):
        return 'Regular'
    elif row['Recency'] >= rfm['Recency'].quantile(0.75) and row['Frequency'] <= rfm['Frequency'].quantile(0.50) and row['Monetary'] <= rfm['Monetary'].quantile(0.50):
        return 'At-Risk'
    else:
        return 'Occasional'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

rfm.head()

In [ ]:
# Segment Count

rfm['Segment'].value_counts()

In [ ]:
# Segment Distribution

plt.figure(figsize=(8,5))
sns.countplot(x=rfm['Segment'])
plt.title('Customer Segment Distribution')
plt.xlabel('Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Customer Cluster Visualization

plt.figure(figsize=(10,6))

sns.scatterplot(
    data=rfm,
    x='Frequency',
    y='Monetary',
    hue='Final_Cluster',
    palette='Set1'
)

plt.title('Customer Segments Based on Frequency and Monetary')
plt.xlabel('Frequency')
plt.ylabel('Monetary')

plt.show()

The scatter plot visualizes customer segments generated by the K-Means clustering algorithm. Each point represents a customer, while colors represent different clusters.

The visualization helps understand how customers are grouped based on their purchasing frequency and spending behavior. Customers with higher frequency and monetary values typically belong to high-value segments, whereas customers with lower values belong to occasional or at-risk segments.

This visualization validates that the clustering algorithm successfully separates customers into meaningful business segments.

**Positive Impact**- Customer cluster visualization enables businesses to clearly identify different customer groups and design targeted marketing strategies. High-value customers can receive loyalty rewards, regular customers can receive personalized recommendations, and at-risk customers can be targeted with retention campaigns.

**Negative Impact**- If customer segments are interpreted incorrectly, marketing resources may be allocated inefficiently. Therefore, cluster profiles should always be analyzed together with business knowledge before making strategic decisions.

### ML Model - 2 (**Agglomerative Hierarchical Clustering**)

In [ ]:
# ML Model - 2 Implementation
# Agglomerative Hierarchical Clustering

from sklearn.cluster import AgglomerativeClustering

agg_model = AgglomerativeClustering(n_clusters=4, linkage='ward')

# Fit the Algorithm and Predict
rfm['Agglomerative_Cluster'] = agg_model.fit_predict(rfm_scaled)

rfm.head()

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

Agglomerative Hierarchical Clustering is an unsupervised clustering algorithm that builds clusters step-by-step by merging similar data points.

It starts with each customer as a separate cluster and gradually combines customers based on similarity. In this project, it was used to segment customers based on RFM behavior.

The model performance was evaluated using Silhouette Score. A higher silhouette score indicates better-defined and more separated clusters.

In [ ]:
# Visualizing evaluation Metric Score chart

agg_silhouette = silhouette_score(rfm_scaled, rfm['Agglomerative_Cluster'])

print("Agglomerative Clustering Silhouette Score:", agg_silhouette)

model_comparison = pd.DataFrame({
    'Model': ['KMeans Clustering', 'Agglomerative Clustering'],
    'Silhouette Score': [
        silhouette_score(rfm_scaled, rfm['Final_Cluster']),
        agg_silhouette
    ]
})

model_comparison

In [ ]:
# Model Comparison Chart

plt.figure(figsize=(8,5))
sns.barplot(x='Model', y='Silhouette Score', data=model_comparison)

plt.title('Model Comparison Based on Silhouette Score')
plt.xlabel('Model')
plt.ylabel('Silhouette Score')
plt.xticks(rotation=20)

plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2 Implementation with hyperparameter optimization techniques

agg_scores = []

for k in range(2, 8):
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = agg.fit_predict(rfm_scaled)
    score = silhouette_score(rfm_scaled, labels)

    agg_scores.append({
        'K': k,
        'Silhouette Score': score
    })

agg_scores_df = pd.DataFrame(agg_scores)

agg_scores_df

In [ ]:
# Agglomerative Clustering Hyperparameter Evaluation

plt.figure(figsize=(10,5))
plt.plot(agg_scores_df['K'], agg_scores_df['Silhouette Score'], marker='o')

plt.title('Agglomerative Clustering Silhouette Scores')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)

plt.show()

In [ ]:
best_agg_k = int(agg_scores_df.sort_values(by='Silhouette Score', ascending=False).iloc[0]['K'])

best_agg_model = AgglomerativeClustering(n_clusters=best_agg_k, linkage='ward')

rfm['Best_Agglomerative_Cluster'] = best_agg_model.fit_predict(rfm_scaled)

print("Best K for Agglomerative Clustering:", best_agg_k)
print("Best Agglomerative Silhouette Score:", silhouette_score(rfm_scaled, rfm['Best_Agglomerative_Cluster']))

##### Which hyperparameter optimization technique have you used and why?

Silhouette Score-based tuning was used for Agglomerative Clustering.

Different values of number of clusters were tested, and the model with the highest silhouette score was selected. This approach is suitable for unsupervised learning because there is no target variable for traditional cross-validation.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, choosing the best cluster count using silhouette score improved the interpretability and separation of customer segments compared to using a fixed cluster count without evaluation.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

**Silhouette Score**:
This metric indicates how well customers are grouped into distinct segments. A higher silhouette score means customers within the same segment are more similar, while customers from different segments are clearly separated.

**Business Impact**:
Well-separated customer segments help businesses create personalized marketing strategies. For example, high-value customers can receive loyalty rewards, regular customers can receive cross-selling offers, and at-risk customers can be targeted with retention campaigns.

**Inertia**:
Inertia measures how compact the clusters are. Lower inertia means customers within a cluster are closer to each other.

**Business Impact**:
Compact clusters improve the accuracy of customer grouping and make segment-based decision-making more reliable.

### ML Model - 3 (**Item-Based Collaborative Filtering Recommendation System**)

In [ ]:
# ML Model - 3 Implementation
# Product Recommendation System using Item-Based Collaborative Filtering

# Create Customer-Product Matrix

customer_product_matrix = df_clean.pivot_table(
    index='CustomerID',
    columns='Description',
    values='Quantity',
    aggfunc='sum',
    fill_value=0
)

customer_product_matrix.head()

In [ ]:
# Compute Product Similarity Matrix

product_similarity = cosine_similarity(customer_product_matrix.T)

product_similarity_df = pd.DataFrame(
    product_similarity,
    index=customer_product_matrix.columns,
    columns=customer_product_matrix.columns
)

product_similarity_df.head()

In [ ]:
# Recommendation Function

def recommend_products(product_name, n=5):
    product_name = product_name.upper()

    matching_products = [
        product for product in product_similarity_df.index
        if product_name in product.upper()
    ]

    if len(matching_products) == 0:
        return ["Product not found. Please try another product name."]

    selected_product = matching_products[0]

    recommendations = product_similarity_df[selected_product].sort_values(ascending=False)[1:n+1]

    return recommendations.index.tolist()

In [ ]:
# Predict on the model

recommend_products('WHITE HANGING HEART T-LIGHT HOLDER', 5)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

Item-Based Collaborative Filtering was used to recommend similar products based on customer purchase history.

A customer-product matrix was created where rows represent customers, columns represent products, and values represent purchased quantities. Cosine similarity was then calculated between products to identify items that are frequently purchased by similar customers.

The system returns the top 5 products most similar to the entered product name.

In [ ]:
# Visualizing evaluation Metric Score chart
# Product Similarity Heatmap for Top 20 Products

top_20_products = df_clean['Description'].value_counts().head(20).index

plt.figure(figsize=(12,8))

sns.heatmap(
    product_similarity_df.loc[top_20_products, top_20_products],
    cmap='coolwarm'
)

plt.title('Product Similarity Heatmap for Top 20 Products')

plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Hyperparameter Tuning
# Testing different numbers of recommendations

recommendation_counts = [3, 5, 7, 10]

sample_product = df_clean['Description'].value_counts().index[0]

for n in recommendation_counts:
    print(f"\nTop {n} recommendations for: {sample_product}")
    print(recommend_products(sample_product, n))

##### Which hyperparameter optimization technique have you used and why?

Manual tuning was used for the recommendation system by testing different values of the number of recommended products.

The project requirement specifies top 5 similar products, so n=5 was selected as the final recommendation count.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The recommendation output became more focused and user-friendly when limited to the top 5 products. A larger number of recommendations may provide more choices but can reduce clarity for the user.

#####Final Model Saving for Streamlit

In [ ]:
# Save models and required files for Streamlit deployment

import pickle

with open('kmeans_model.pkl', 'wb') as file:
    pickle.dump(best_kmeans, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

with open('product_similarity.pkl', 'wb') as file:
    pickle.dump(product_similarity_df, file)

with open('rfm_data.pkl', 'wb') as file:
    pickle.dump(rfm, file)

print("All models and files saved successfully.")

In [ ]:
from google.colab import files

files.download('kmeans_model.pkl')
files.download('scaler.pkl')
files.download('product_similarity.pkl')
files.download('rfm_data.pkl')

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Silhouette Score and Inertia were considered for customer segmentation.

Silhouette Score was important because it measures how well customers are separated into meaningful clusters. Better separation allows the business to create targeted marketing strategies for different customer groups.

Inertia was considered because it measures cluster compactness. Compact clusters help ensure customers within the same segment have similar purchasing behavior.

For the recommendation system, cosine similarity was used because it identifies products that are frequently purchased by similar customers.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

K-Means Clustering was selected as the final customer segmentation model because it produced interpretable customer groups using RFM features and performed well based on inertia and silhouette score analysis.

For product recommendations, Item-Based Collaborative Filtering was selected because it effectively recommends similar products based on customer purchase history.

Therefore, the final solution uses K-Means for customer segmentation and Collaborative Filtering for product recommendation.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

The final segmentation model uses K-Means Clustering on three important RFM features: Recency, Frequency, and Monetary.

Recency indicates how recently a customer purchased.
Frequency indicates how often a customer purchased.
Monetary indicates how much a customer spent.

These features are important because they directly represent customer value and purchasing behavior.

In unsupervised clustering, traditional feature importance methods such as SHAP are not directly applicable because there is no target variable. Therefore, feature importance was interpreted using business logic and cluster profiles.

# **Conclusion**



The Shopper Spectrum project successfully analyzed e-commerce transaction data to understand customer purchasing behavior and generate actionable business insights. Through comprehensive data preprocessing, exploratory data analysis, feature engineering, customer segmentation, and product recommendation modeling, valuable patterns were identified that can support data-driven business decisions.

The dataset was cleaned by removing missing customer records, cancelled transactions, and invalid purchase entries. Exploratory Data Analysis revealed significant trends in customer activity, country-wise revenue contribution, product demand, and spending behavior. The United Kingdom emerged as the dominant market in terms of both revenue and transaction volume, while a small number of products and customers contributed a significant portion of total sales.

RFM (Recency, Frequency, Monetary) analysis was performed to capture customer purchasing behavior. Using K-Means Clustering and Agglomerative Clustering, customers were successfully segmented into meaningful groups such as High-Value, Regular, Occasional, and At-Risk customers. These segments provide businesses with opportunities to design personalized marketing campaigns, improve customer retention, and increase customer lifetime value.

In addition, an Item-Based Collaborative Filtering Recommendation System was developed using cosine similarity. The system recommends products based on customer purchase patterns and enables personalized shopping experiences. Such recommendations can improve customer satisfaction, increase cross-selling opportunities, and drive additional revenue.

Overall, the project demonstrates how machine learning and data analytics can be leveraged to transform raw transaction data into valuable business intelligence. The combination of customer segmentation and product recommendation provides a scalable solution for enhancing customer engagement, optimizing marketing strategies, improving inventory planning, and supporting sustainable business growth in the e-commerce industry.

# **Future Scope**


• Implement real-time recommendation systems using live transaction data.

• Integrate advanced recommendation techniques such as Matrix Factorization and Deep Learning-based Recommender Systems.

• Develop automated customer retention campaigns based on customer segment predictions.

• Incorporate customer demographics and browsing behavior for more accurate segmentation.

• Deploy the solution as a production-ready Streamlit web application for business users.

• Enable dynamic inventory forecasting based on customer demand patterns and purchasing trends.

• Integrate the recommendation engine with e-commerce platforms to provide personalized product suggestions in real time.Implement real-time recommendation systems using live transaction data.

• Integrate advanced recommendation techniques such as Matrix Factorization and Deep Learning-based Recommender Systems.

• Develop automated customer retention campaigns based on customer segment predictions.

• Incorporate customer demographics and browsing behavior for more accurate segmentation.

• Deploy the solution as a production-ready Streamlit web application for business users.

• Enable dynamic inventory forecasting based on customer demand patterns and purchasing trends.

• Integrate the recommendation engine with e-commerce platforms to provide personalized product suggestions in real time.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***